# 05 - Deploy the data agent

Deploys **and publishes** the *Fabric Arch Review - Data Agent* and (optionally) runs its
self-evaluation. This is a standalone notebook - it is **not** part of the pipeline, because
the `fabric-data-agent-sdk` needs a `%pip install` + kernel restart that an automated pipeline
run can't do.

**Run order:** run the pipeline once (so the gold tables exist and are populated), then run this
notebook: run the install cell below, let Fabric **restart the kernel** if it asks, then run the
deploy cell. The agent grounds on the gold tables via the SDK, which lets the Fabric backend
*enumerate* each datasource's schema - the only reliable way to reference tables (a hand-authored
table list gets flagged *"table has been deleted or you don't have permission"*).


In [ ]:
# Install the tested Fabric data agent SDK preview. If Fabric prompts to
# RESTART THE KERNEL, do it, then run the deploy cell below.
%pip install -q fabric-data-agent-sdk==0.1.30a0

In [ ]:
# Deploy + publish the data agent, then run its deterministic self-eval.
# Self-contained so it works after the kernel restart from the %pip cell above.
import os, sys, shutil, subprocess, datetime as dt

# ---- parameters ----
GITHUB_REPO_URL = "https://github.com/microsoft/fabric-architecture-review.git"
GITHUB_BRANCH   = "main"
GITHUB_REF      = ""          # pinned tag (e.g. v2026.07.0); blank = branch tip
DATA_AGENT_NAME     = "Fabric Arch Review - Data Agent"
SEMANTIC_MODEL_NAME = "Fabric Arch Review - Governance"
LAKEHOUSE_NAME      = "fabric_arch_review_lh"
RUN_ID          = "latest"    # which run's gold to score the agent against
EVALUATE_AGENT  = "true"      # "false" to deploy only (skip writing gold_agent_eval)

# ---- clone the framework (fresh, so it survives the %pip restart) ----
WORK_ROOT = "/tmp/fabric-arch-review-agent"
REPO_DIR  = os.path.join(WORK_ROOT, "repo")
os.makedirs(WORK_ROOT, exist_ok=True)
_url = GITHUB_REPO_URL
_ref = (GITHUB_REF or "").strip() or GITHUB_BRANCH
if os.path.isdir(REPO_DIR):
    shutil.rmtree(REPO_DIR)
subprocess.run(["git", "clone", "--branch", _ref, "--depth", "1", _url, REPO_DIR], check=True)
del _url
if REPO_DIR not in sys.path:
    sys.path.insert(0, REPO_DIR)
os.chdir(REPO_DIR)
subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-r", os.path.join(REPO_DIR, "requirements.txt")])

_far_ver = ""
try:
    with open(os.path.join(REPO_DIR, "VERSION"), encoding="utf-8-sig") as _vf:
        _far_ver = _vf.read().strip()
except Exception:
    pass

# ---- deploy + publish the agent (backend enumerates the schema -> no phantom tables) ----
from reports.agent.sdk_deploy import deploy_agent
deploy_agent(
    agent_name=DATA_AGENT_NAME,
    model_name=SEMANTIC_MODEL_NAME,
    lakehouse_name=LAKEHOUSE_NAME,
    version=_far_ver,
    publish=True,
)
print("Data agent deployed + published:", DATA_AGENT_NAME)

# ---- deterministic self-eval: ask the published agent, score vs gold, write gold_agent_eval ----
if EVALUATE_AGENT == "true":
    try:
        from reports.gold_layer import build_gold_from_dir
        from reports.agent.evaluate import make_fabric_ask, run_evaluation, summarize
        from reports.powerbi.schema import GOLD_TABLES_BY_NAME
        from pyspark.sql import SparkSession
        from pyspark.sql.types import (StructType, StructField, StringType, LongType,
                                       DoubleType, BooleanType, TimestampType)

        LH = "/lakehouse/default/Files"
        _base = os.path.join(LH, "fabric-arch-review")
        _run = RUN_ID
        if _run == "latest":
            _runs = [d for d in os.listdir(_base) if os.path.isdir(os.path.join(_base, d, "raw"))]
            _run = max(_runs, key=lambda d: os.path.getmtime(os.path.join(_base, d)))
        _out = os.path.join(_base, _run)
        _ts = dt.datetime.now(dt.timezone.utc).replace(microsecond=0, tzinfo=None).isoformat() + "Z"

        _tables = build_gold_from_dir(_out, run_id=_run, run_timestamp=_ts, check_remote=False)
        _ask = make_fabric_ask(DATA_AGENT_NAME)
        _rows = run_evaluation(_ask, _tables, run_id=_run, run_timestamp=_ts)
        print("agent eval:", summarize(_rows))

        spark = SparkSession.builder.getOrCreate()
        _T = {"string": StringType(), "int64": LongType(), "double": DoubleType(),
              "boolean": BooleanType(), "dateTime": TimestampType()}
        _t = GOLD_TABLES_BY_NAME["gold_agent_eval"]
        _sch = StructType([StructField(c.name, _T[c.kind], True) for c in _t.columns])

        def _to_row(r):
            rec = []
            for c in _t.columns:
                v = r.get(c.name)
                if c.kind == "dateTime" and isinstance(v, str) and v:
                    v = dt.datetime.fromisoformat(v.replace("Z", "+00:00")).replace(tzinfo=None)
                rec.append(v)
            return tuple(rec)

        (spark.createDataFrame([_to_row(r) for r in _rows], schema=_sch)
            .write.format("delta").mode("append").option("mergeSchema", "true")
            .saveAsTable("gold_agent_eval"))
        print("gold_agent_eval: wrote", len(_rows), "row(s) - the report's Agent Eval page will populate.")
    except Exception as _e:
        print("agent self-eval skipped (best-effort):", _e)
